导入fashion mnist

In [2]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_data = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_data, batch_size=256, shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_data, batch_size=256, shuffle=False,
    num_workers=0
)

x, y = next(iter(train_loader))
print(x.shape, y.shape)

torch.Size([256, 1, 28, 28]) torch.Size([256])


In [3]:
net = nn.Sequential(
    nn.Conv2d(
        in_channels=1,
        out_channels=6,
        kernel_size=5,
        padding=2
    ),
    nn.Sigmoid(),
    nn.AvgPool2d(
        kernel_size=2,
        stride=2
    ),
    nn.Conv2d(
        in_channels=6,
        out_channels=16,
        kernel_size=5,
        padding=0
    ),
    nn.Sigmoid(),
    nn.AvgPool2d(
        kernel_size=2,
        stride=2
    ),
    nn.Flatten(),
    nn.Linear(400,120),
    nn.Sigmoid(),
    nn.Linear(120,84),
    nn.Sigmoid(),
    nn.Linear(84,10)
)

x_test = torch.randn(1,1,28,28)
y = net(x_test)
for layer in net:
    x_test = layer(x_test)
    print(layer)
    print(x_test.shape)
print(sum(p.numel() for p in net.parameters()))

Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
torch.Size([1, 6, 28, 28])
Sigmoid()
torch.Size([1, 6, 28, 28])
AvgPool2d(kernel_size=2, stride=2, padding=0)
torch.Size([1, 6, 14, 14])
Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
torch.Size([1, 16, 10, 10])
Sigmoid()
torch.Size([1, 16, 10, 10])
AvgPool2d(kernel_size=2, stride=2, padding=0)
torch.Size([1, 16, 5, 5])
Flatten(start_dim=1, end_dim=-1)
torch.Size([1, 400])
Linear(in_features=400, out_features=120, bias=True)
torch.Size([1, 120])
Sigmoid()
torch.Size([1, 120])
Linear(in_features=120, out_features=84, bias=True)
torch.Size([1, 84])
Sigmoid()
torch.Size([1, 84])
Linear(in_features=84, out_features=10, bias=True)
torch.Size([1, 10])
61706


In [5]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

net = net.to(device)

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    net.parameters(), lr=1e-2
)

epochs = 30

for epoch in range(epochs):
    net.train()

    loss_sum = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        batch_size = x.shape[0]
        optimizer.zero_grad()
        y_hat = net(x)
        l = loss(y_hat,y)
        l.backward()
        optimizer.step()
        loss_sum += l.item()*batch_size
        pred = torch.argmax(y_hat,dim=1)
        correct += (pred==y).sum().item()
        total += batch_size
        
    train_loss = loss_sum / total
    train_acc = correct / total

    net.eval()

    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            y_hat = net(x)
            pred = torch.argmax(y_hat,dim=1)
            test_correct += (pred==y).sum().item()
            test_total += x.shape[0]

    test_acc = test_correct / test_total

    print(
        f"Epoch {epoch+1}: "
        f"loss={train_loss:.4f}, "
        f"train_acc={train_acc:.4f}, "
        f"test_acc={test_acc:.4f}"
    )

Epoch 1: loss=0.5334, train_acc=0.8001, test_acc=0.8249
Epoch 2: loss=0.4386, train_acc=0.8382, test_acc=0.8413
Epoch 3: loss=0.3977, train_acc=0.8518, test_acc=0.8445
Epoch 4: loss=0.3675, train_acc=0.8634, test_acc=0.8560
Epoch 5: loss=0.3451, train_acc=0.8702, test_acc=0.8577
Epoch 6: loss=0.3258, train_acc=0.8774, test_acc=0.8614
Epoch 7: loss=0.3102, train_acc=0.8830, test_acc=0.8786
Epoch 8: loss=0.3013, train_acc=0.8880, test_acc=0.8823
Epoch 9: loss=0.2880, train_acc=0.8924, test_acc=0.8861
Epoch 10: loss=0.2800, train_acc=0.8950, test_acc=0.8823
Epoch 11: loss=0.2736, train_acc=0.8978, test_acc=0.8849
Epoch 12: loss=0.2638, train_acc=0.9017, test_acc=0.8889
Epoch 13: loss=0.2566, train_acc=0.9034, test_acc=0.8903
Epoch 14: loss=0.2467, train_acc=0.9072, test_acc=0.8883
Epoch 15: loss=0.2415, train_acc=0.9097, test_acc=0.8907
Epoch 16: loss=0.2315, train_acc=0.9127, test_acc=0.8895
Epoch 17: loss=0.2306, train_acc=0.9130, test_acc=0.8945
Epoch 18: loss=0.2241, train_acc=0.9165,